In [0]:

from functools import reduce

from pyspark.sql import functions as F

SILVER_SCHEMA = "football_data_platform.silver"

fbref_tables = [
    "team_standard_stats",
    "team_shooting_stats",
    "team_goalkeeping_stats",
    "team_playing_time_stats",
    "team_miscellaneous_stats",
    "team_standings_stats",
]

api_tables = [
    "football_data_org_competitions",
    "football_data_org_teams",
    "football_data_org_matches",
    "football_data_org_standings",
]

other_tables = [
    "match_results_football_data",
    "canonical_teams",
    "team_mappings",
]

all_tables = fbref_tables + api_tables + other_tables

# Gold candidate keys, not necessarily the existing
# Silver deduplication keys.
gold_candidate_keys = {
    **{
        table: [
            "team_key",
            "season_start_year",
            "league_code",
        ]
        for table in fbref_tables
    },
    "football_data_org_competitions": [
        "competition_id",
    ],
    "football_data_org_teams": [
        "competition_id",
        "season_id",
        "team_id",
    ],
    "football_data_org_matches": [
        "match_id",
    ],
    "football_data_org_standings": [
        "competition_id",
        "season_id",
        "standing_type",
        "stage",
        "standing_group",
        "team_id",
    ],
    "match_results_football_data": [
        "season_code",
        "league_name",
        "match_date",
        "home_team_key",
        "away_team_key",
    ],
    "canonical_teams": [
        "team_key",
    ],
}

# Required identity columns to check for missing values.
required_columns = {
    **{
        table: [
            "team_key",
            "season_start_year",
            "league_code",
        ]
        for table in fbref_tables
    },
    "football_data_org_competitions": [
        "competition_id",
        "league_code",
    ],
    "football_data_org_teams": [
        "team_id",
        "team_key",
        "season_start_year",
        "league_code",
    ],
    "football_data_org_matches": [
        "match_id",
        "home_team_key",
        "away_team_key",
        "season_start_year",
        "league_code",
    ],
    "football_data_org_standings": [
        "team_id",
        "team_key",
        "season_start_year",
        "league_code",
    ],
    "match_results_football_data": [
        "match_date",
        "home_team_key",
        "away_team_key",
        "season_start_year",
    ],
    "canonical_teams": [
        "team_key",
        "team_name",
    ],
    "team_mappings": [
        "team_key",
        "source_system",
    ],
}

profile_results = []
schema_results = []

for table_name in all_tables:

    full_name = f"{SILVER_SCHEMA}.{table_name}"

    print(f"Profiling: {full_name}")

    df = spark.table(full_name)
    available_columns = set(df.columns)
    row_count = df.count()

    keys = gold_candidate_keys.get(table_name, [])
    missing_keys = [
        col for col in keys if col not in available_columns
    ]

    if keys and not missing_keys:
        duplicate_groups = (
            df.groupBy(*keys)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
    else:
        duplicate_groups = None

    required = required_columns.get(table_name, [])
    missing_required = [
        col for col in required
        if col not in available_columns
    ]

    present_required = [
        col for col in required
        if col in available_columns
    ]

    null_counts = {}

    if present_required:
        aggregations = [
            F.sum(
                F.when(F.col(col).isNull(), 1)
                .otherwise(0)
            ).alias(col)
            for col in present_required
        ]

        null_counts = (
            df.agg(*aggregations)
            .first()
            .asDict()
        )

    profile_results.append({
        "table_name": table_name,
        "row_count": row_count,
        "column_count": len(df.columns),
        "duplicate_key_groups": duplicate_groups,
        "null_counts": str(null_counts),
        "missing_key_columns": str(missing_keys),
        "missing_required_columns": str(missing_required),
    })

    for field in df.schema.fields:
        schema_results.append({
            "table_name": table_name,
            "column_name": field.name,
            "data_type": field.dataType.simpleString(),
            "nullable": field.nullable,
        })

profile_df = spark.createDataFrame(profile_results)
schema_df = spark.createDataFrame(schema_results)

display(profile_df.orderBy("table_name"))
display(schema_df.orderBy("table_name", "column_name"))


In [0]:

coverage_tables = (
    fbref_tables
    + [
        "football_data_org_matches",
        "football_data_org_teams",
        "football_data_org_standings",
        "match_results_football_data",
    ]
)

coverage_dfs = []

for table_name in coverage_tables:

    df = spark.table(
        f"{SILVER_SCHEMA}.{table_name}"
    )

    league_column = (
        "league_code"
        if "league_code" in df.columns
        else "league_name"
    )

    coverage = (
        df.groupBy(
            F.col(league_column).alias("competition"),
            "season_start_year",
        )
        .agg(F.count("*").alias("records"))
        .withColumn("source_table", F.lit(table_name))
    )

    coverage_dfs.append(coverage)

coverage_df = reduce(
    lambda a, b: a.unionByName(b),
    coverage_dfs,
)

display(
    coverage_df.orderBy(
        "season_start_year",
        "competition",
        "source_table",
    )
)


In [0]:

from pyspark.sql import functions as F

SILVER = "football_data_platform.silver"

fbref_tables = [
    "team_standard_stats",
    "team_shooting_stats",
    "team_goalkeeping_stats",
    "team_playing_time_stats",
    "team_miscellaneous_stats",
    "team_standings_stats",
]

KEYS = [
    "team_key",
    "league_code",
    "season_start_year",
]

base = (
    spark.table(f"{SILVER}.team_standard_stats")
    .select(*KEYS)
    .distinct()
)

results = []

for table_name in fbref_tables:

    current = (
        spark.table(f"{SILVER}.{table_name}")
        .select(*KEYS)
        .distinct()
    )

    missing_from_current = base.join(
        current, KEYS, "left_anti"
    ).count()

    extra_in_current = current.join(
        base, KEYS, "left_anti"
    ).count()

    results.append({
        "table_name": table_name,
        "missing_keys": missing_from_current,
        "extra_keys": extra_in_current,
        "keys_match": (
            missing_from_current == 0
            and extra_in_current == 0
        ),
    })

display(spark.createDataFrame(results))


In [0]:

from pyspark.sql import functions as F

canonical = (
    spark.table(f"{SILVER}.canonical_teams")
    .select("team_key")
    .distinct()
)

team_key_sources = {
    **{
        name: ["team_key"]
        for name in fbref_tables
    },
    "football_data_org_teams": ["team_key"],
    "football_data_org_standings": ["team_key"],
    "football_data_org_matches": [
        "home_team_key",
        "away_team_key",
    ],
    "match_results_football_data": [
        "home_team_key",
        "away_team_key",
    ],
    "team_mappings": ["team_key"],
}

results = []
orphan_details = []

for table_name, columns in team_key_sources.items():

    df = spark.table(f"{SILVER}.{table_name}")

    for column in columns:

        orphan_keys = (
            df.select(F.col(column).alias("team_key"))
            .distinct()
            .join(canonical, "team_key", "left_anti")
        )

        # Include nulls as failures if present.
        orphan_count = orphan_keys.count()

        results.append({
            "table_name": table_name,
            "column_name": column,
            "orphan_distinct_keys": orphan_count,
        })

        if orphan_count > 0:
            examples = orphan_keys.limit(20).collect()

            for row in examples:
                orphan_details.append({
                    "table_name": table_name,
                    "column_name": column,
                    "team_key": row["team_key"],
                })

display(spark.createDataFrame(results))

if orphan_details:
    display(spark.createDataFrame(orphan_details))


In [0]:

from pyspark.sql import functions as F

KEYS = [
    "season_start_year",
    "league_code",
    "match_date",
    "home_team_key",
    "away_team_key",
]

csv_matches = (
    spark.table(
        f"{SILVER}.match_results_football_data"
    )
    .filter(F.col("season_start_year") == 2026)
    .select(
        F.col("league_name").alias("league_code"),
        "season_start_year",
        "match_date",
        "home_team_key",
        "away_team_key",
        "full_time_home_goals",
        "full_time_away_goals",
    )
)

api_matches = (
    spark.table(
        f"{SILVER}.football_data_org_matches"
    )
    .filter(F.col("season_start_year") == 2026)
    .select(
        *KEYS,
        "match_id",
        "status",
        "home_goals",
        "away_goals",
    )
)

matched = csv_matches.join(
    api_matches,
    KEYS,
    "inner",
)

unmatched_csv = csv_matches.join(
    api_matches.select(*KEYS),
    KEYS,
    "left_anti",
)

score_mismatches = matched.filter(
    ~F.col("full_time_home_goals").eqNullSafe(
        F.col("home_goals")
    )
    |
    ~F.col("full_time_away_goals").eqNullSafe(
        F.col("away_goals")
    )
)

print("CSV records:", csv_matches.count())
print("API records:", api_matches.count())
print("Exact matches:", matched.count())
print("Unmatched CSV:", unmatched_csv.count())
print("Score mismatches:", score_mismatches.count())

print("API match statuses:")
display(
    api_matches.groupBy("status")
    .count()
    .orderBy("status")
)

print("Unmatched CSV matches:")
display(unmatched_csv.orderBy("match_date"))

print("Score mismatches:")
display(score_mismatches.orderBy("match_date"))


In [0]:

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Inspect Nantes and Toulouse across FBref Silver datasets
# ---------------------------------------------------------------------------

SILVER = "football_data_platform.silver"

FBREF_TABLES = [
    "team_standard_stats",
    "team_shooting_stats",
    "team_goalkeeping_stats",
    "team_playing_time_stats",
    "team_miscellaneous_stats",
    "team_standings_stats",
]

TEAM_KEYS = [90, 126]

results = []

for table_name in FBREF_TABLES:

    df = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            (F.col("season_start_year") == 2025)
            & (F.col("league_code") == "ligue_one")
            & F.col("team_key").isin(TEAM_KEYS)
        )
    )

    columns = [
        "team_key",
        "matches_played",
        "_silver_processed_at",
    ]

    selected = [
        col for col in columns
        if col in df.columns
    ]

    for row in df.select(*selected).collect():

        results.append({
            "source_table": table_name,
            "team_key": row["team_key"],
            "matches_played": (
                row["matches_played"]
                if "matches_played" in row.asDict()
                else None
            ),
            "silver_processed_at": str(
                row["_silver_processed_at"]
            ) if "_silver_processed_at" in row.asDict()
            else None,
        })

display(
    spark.createDataFrame(results)
    .orderBy("team_key", "source_table")
)
